# Client Clustering and Profile Analysis

The goal of this notebook is to group our 370 clients based on their consumption behavior (patterns) rather than their total volume. This allows us to train specialized models for specific groups (e.g., industrial vs. residential).

## 1. Imports

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "2"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

## 2. Hourly Profile Aggregation
We calculate the mean consumption for each of the 24 hours of the day across the entire dataset for every client. This results in a 370x24 matrix, where each row represents a client's "average day."

In [ ]:
# ---------------------------------------------------------
# Load Data and Build Hourly Profiles
# ---------------------------------------------------------
print("Loading processed data...")
df = pd.read_parquet('../Datasets/processed_electricity_data.parquet')

print("Creating average hourly profiles...")
profiles = df.groupby(['ClientID', 'Hour'], observed=True)['Consumption'].mean().unstack()

## 3. Scaling by Row
Standardization is vital here. By applying a Min-Max scaler to each individual client, we remove the "volume" effect. This allows the K-Means algorithm to group a small shop and a large supermarket together if they both share a "9-to-5" consumption signature.

In [ ]:
# ---------------------------------------------------------
# Row-wise Normalization and Elbow Method
# ---------------------------------------------------------
print("Normalizing profiles...")
scaler = MinMaxScaler()
profiles_scaled = scaler.fit_transform(profiles.T).T

inertia = []
K_range = range(1, 11)

print("Calculating inertia for the Elbow Method...")
for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(profiles_scaled)
    inertia.append(model.inertia_)

plt.figure(figsize=(12, 6))
plt.plot(K_range, inertia, marker='o', linestyle='-', color='b')
plt.xlabel('Number of Clusters (k)', fontsize=12)
plt.ylabel('Inertia (WCSS)', fontsize=12)
plt.title('Elbow Method for Optimal k', fontsize=14)
plt.xticks(K_range)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ---------------------------------------------------------
# K-Means Clustering (k=5)
# ---------------------------------------------------------
n_clusters = 5
print(f"Running K-Means with {n_clusters} clusters...")
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(profiles_scaled)

mapping = pd.DataFrame({
    'ClientID': profiles.index,
    'Cluster':  cluster_labels
})

## 4. Interpretation
The resulting plot helps us identify consumer types:

- **Cluster 0 (Blue - 14 clients)**: Night-Shift Industrial. This unique group shows peak consumption from midnight to 6:00 AM, with a significant drop during daylight hours.

- **Cluster 1 (Orange - 81 clients)**: Standard Daytime Business. Features a steady rise starting at 8:00 AM, peaking in the afternoon, and maintaining high activity until late evening (around 9:00 PM).

- **Cluster 2 (Green - 78 clients)**: Late Evening Residential. This group shows a massive spike starting at 7:00 PM and peaking at 9:00 PM, likely representing residential areas with high night-time activity.

- **Cluster 3 (Red - 34 clients)**: Split-Shift/Siesta Profile. These clients have a sharp peak at noon followed by a visible dip at 2:00 PM, a classic "siesta" signature common in Portuguese commercial districts.

- **Cluster 4 (Purple - 163 clients)**: General Commercial/Residential. Representing nearly half the clients, this cluster follows a standard "early rise" profile with high, sustained consumption from 10:00 AM through 8:00 PM.

In [ ]:
# ---------------------------------------------------------
# Cluster Profile Visualization
# ---------------------------------------------------------
plt.figure(figsize=(12, 6))
for i in range(n_clusters):
    cluster_mean = profiles_scaled[cluster_labels == i].mean(axis=0)
    plt.plot(range(1, 25), cluster_mean, label=f'Cluster {i} (n={sum(cluster_labels == i)})', linewidth=2)

plt.title('Normalized Daily Load Profiles by Cluster', fontsize=14)
plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Normalized Consumption (0-1)', fontsize=12)
plt.xticks(range(1, 25))
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ---------------------------------------------------------
# Save Cluster Mapping
# ---------------------------------------------------------
print("Saving cluster mapping...")
mapping.to_csv('../Datasets/client_clusters.csv', index=False)